# 03 — LightGBM (log-loss focus)

Locked for every electrum notebook:

| Rule | Value |
|------|--------|
| Train | 2020–2022 |
| Val (selection) | 2023 |
| Test | **sealed** — do not load `*_test.parquet` |
| Split shuffle | **No** (calendar splits only) |
| 50/50 resampling / SMOTE | **No** |
| PCA on trees | **No** |
| Headline score | **PR-AUC** (val chance ≈ 0.041; Strong ≥ 0.10) |
| Accuracy | Logged on train/val curves only — a bad headline here (always-stay wins) |

CUDA on this Windows machine is **WSL2**, not native PowerShell. Trees: CPU. TabM / FT-T: torch CUDA if available. NASNet: TF GPU via WSL; warn instead of a silent CPU marathon.

```text
CPU:  .\.venv\Scripts\Activate.ps1  then jupyter
GPU:  .\scripts\wsl-python.cmd -m jupyter notebook electrum/03_lightgbm.ipynb
```

Matrix: `tree_v1`. Trainer: `src/modeling/train_lightgbm.py`.

## Setup

In [1]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

REPO = Path.cwd().resolve()
for p in [REPO, *REPO.parents]:
    if (p / "src" / "modeling").exists():
        REPO = p
        break
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

MODEL_ID = "lightgbm"
RESULTS = REPO / "electrum" / "results" / MODEL_ID
(RESULTS / "models").mkdir(parents=True, exist_ok=True)
(RESULTS / "plots").mkdir(parents=True, exist_ok=True)

CHANCE_PR = 0.041
STRONG_PR = 0.10


def probe_cuda() -> dict:
    report: dict = {}
    try:
        import torch

        report["torch"] = torch.__version__
        report["torch_cuda"] = bool(torch.cuda.is_available())
        report["torch_device"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    except Exception as e:  # noqa: BLE001
        report["torch"] = f"unavailable: {e}"
        report["torch_cuda"] = False
        report["torch_device"] = "cpu"
    try:
        import tensorflow as tf

        gpus = tf.config.list_physical_devices("GPU")
        report["tf"] = tf.__version__
        report["tf_gpus"] = [g.name for g in gpus]
        report["tf_built_with_cuda"] = bool(tf.test.is_built_with_cuda())
        # Keep TF from grabbing the whole card so later torch training can use CUDA.
        # NASNet re-enables TF GPU in its train cell via require_gpu().
        if MODEL_ID != "nasnet_cnn" and gpus:
            try:
                tf.config.set_visible_devices([], "GPU")
                report["tf_gpu_hidden_for_torch"] = True
            except Exception as e:  # noqa: BLE001
                report["tf_gpu_hidden_for_torch"] = f"failed: {e}"
    except Exception as e:  # noqa: BLE001
        report["tf"] = f"unavailable: {e}"
        report["tf_gpus"] = []
    print(json.dumps(report, indent=2))
    if not report.get("torch_cuda") and not report.get("tf_gpus"):
        print("No CUDA visible in this kernel. Trees are fine on CPU. For TabM/FT-T/NASNet use WSL: scripts/wsl-python.cmd")
    return report


CUDA = probe_cuda()

{
  "torch": "2.14.0+cpu",
  "torch_cuda": false,
  "torch_device": "cpu",
  "tf": "2.21.0",
  "tf_gpus": [],
  "tf_built_with_cuda": false
}
No CUDA visible in this kernel. Trees are fine on CPU. For TabM/FT-T/NASNet use WSL: scripts/wsl-python.cmd


## Knobs

In [2]:
from src.modeling.retrain_ameliorations import LGBM_LOGLOSS_FOCUS

KNOBS = {
    "SEED": 2026,
    "USE_CLASS_WEIGHT": False,  # scale_pos_weight ~ train 88/12 ≈ 7
    "DROP_SPARSE": False,
    "LOGLOSS_FOCUS": True,      # stronger L2 / shallower trees vs default HPO PR-AUC peak
}
print(KNOBS)

{'SEED': 2026, 'USE_CLASS_WEIGHT': False, 'DROP_SPARSE': False, 'LOGLOSS_FOCUS': True}


In [3]:
def _public_metrics(m: dict) -> dict:
    skip = {"history"}
    out = {}
    for k, v in m.items():
        if str(k).startswith("_") or k in skip:
            continue
        if isinstance(v, (np.floating, np.integer)):
            out[k] = float(v)
        elif isinstance(v, (float, int, str, bool)) or v is None:
            out[k] = v
        elif isinstance(v, (list, tuple)) and len(v) <= 50:
            out[k] = list(v)
    return out


def history_frame(m: dict) -> pd.DataFrame:
    hist = m.get("history") or m.get("_history") or []
    if not hist:
        return pd.DataFrame(columns=["step", "train_loss", "train_acc", "val_loss", "val_acc", "val_pr_auc"])
    return pd.DataFrame(hist)


def _safe_show(fig) -> None:
    try:
        fig.show()
    except Exception as e:  # noqa: BLE001
        print("plotly show skipped (headless):", e)


def plot_metrics_bar(m: dict):
    raw = m.get("pr_auc")
    pr = float(raw) if raw is not None and raw == raw else float("nan")
    fig = go.Figure(
        data=[
            go.Bar(name="this run", x=["PR-AUC"], y=[pr]),
            go.Bar(name="chance 0.041", x=["PR-AUC"], y=[CHANCE_PR]),
            go.Bar(name="Strong 0.10", x=["PR-AUC"], y=[STRONG_PR]),
        ]
    )
    fig.update_layout(title="Val PR-AUC vs chance / Strong (test sealed)", barmode="group", yaxis_title="PR-AUC")
    try:
        fig.write_html(RESULTS / "plots" / "val_pr_auc.html")
    except Exception as e:  # noqa: BLE001
        print("pr-auc html skipped:", e)
    _safe_show(fig)
    return fig


def plot_history(df: pd.DataFrame):
    if df.empty:
        print("No epoch/iteration history (single-fit family). metrics.json still written.")
        return None
    fig = go.Figure()
    for col, name in (
        ("train_loss", "train loss"),
        ("val_loss", "val loss"),
        ("train_acc", "train acc"),
        ("val_acc", "val acc"),
    ):
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df.get("step", df.get("epoch")), y=df[col], mode="lines", name=name))
    fig.update_layout(
        title="Train/val loss and accuracy (accuracy is not the headline; PR-AUC is)",
        xaxis_title="step (iteration ≈ epoch)",
        yaxis_title="value",
    )
    try:
        fig.write_html(RESULTS / "plots" / "history.html")
    except Exception as e:  # noqa: BLE001
        print("history html skipped:", e)
    _safe_show(fig)
    return fig


def append_leaderboard(m: dict) -> None:
    path = REPO / "electrum" / "results" / "leaderboard.csv"
    row = {
        "model": MODEL_ID,
        "run_name": m.get("model", MODEL_ID),
        "split": "val",
        "status": m.get("status"),
        "pr_auc": m.get("pr_auc"),
        "pr_lift": m.get("pr_lift"),
        "roc_auc": m.get("roc_auc"),
        "log_loss": m.get("log_loss"),
        "brier": m.get("brier"),
        "precision_at_10pct": m.get("precision_at_10pct"),
        "withdrawn_mw_capture_at_10pct": m.get("withdrawn_mw_capture_at_10pct"),
        "n_features": m.get("n_features"),
        "reason": m.get("reason"),
    }
    df = pd.DataFrame([row])
    if path.exists() and path.stat().st_size > 0:
        old = pd.read_csv(path)
        old = old[old["model"] != MODEL_ID]
        df = pd.concat([old, df], ignore_index=True)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print("leaderboard →", path)


def save_bundle(m: dict, model_obj=None, model_filename: str = "model.joblib") -> None:
    m = m or {"model": MODEL_ID, "status": "skipped", "reason": "no metrics dict"}
    pub = _public_metrics(m)
    pub.setdefault("model", m.get("model", MODEL_ID))
    pub.setdefault("status", m.get("status", "ok"))
    (RESULTS / "metrics.json").write_text(json.dumps(pub, indent=2, default=str), encoding="utf-8")
    hist = history_frame(m)
    hist.to_csv(RESULTS / "history.csv", index=False)
    if model_obj is not None and str(pub.get("status")) == "ok":
        try:
            import joblib

            joblib.dump(model_obj, RESULTS / "models" / model_filename)
            print("model →", RESULTS / "models" / model_filename)
        except Exception as e:  # noqa: BLE001
            print("model save skipped:", e)
    try:
        plot_metrics_bar(m)
    except Exception as e:  # noqa: BLE001
        print("metrics bar skipped:", e)
    try:
        plot_history(hist)
    except Exception as e:  # noqa: BLE001
        print("history plot skipped:", e)
    append_leaderboard(pub)
    print("status=", pub.get("status"), "wrote", RESULTS)


def skipped(reason: str) -> dict:
    return {"model": MODEL_ID, "status": "skipped", "reason": str(reason), "split": "val"}

## Load + train

In [4]:
from src.modeling.retrain_ameliorations import SPARSE_CANDIDATES
from src.modeling.train_lightgbm import fit_lightgbm_eval

try:
    params = dict(LGBM_LOGLOSS_FOCUS) if KNOBS["LOGLOSS_FOCUS"] else {}
    params["random_state"] = int(KNOBS["SEED"])
    if KNOBS["USE_CLASS_WEIGHT"]:
        params["scale_pos_weight"] = 7.0
    drop = SPARSE_CANDIDATES if KNOBS["DROP_SPARSE"] else None
    metrics = fit_lightgbm_eval(params=params, drop_cols=drop, model_name="lightgbm_electrum", save=False)
except Exception as e:  # noqa: BLE001
    metrics = skipped(f"{type(e).__name__}: {e}")
    print("TRAIN FAILED (recorded as skip):", metrics["reason"])
print({k: metrics.get(k) for k in ("status", "pr_auc", "roc_auc", "log_loss", "withdrawn_mw_capture_at_10pct", "best_iteration", "reason")})

{'status': 'ok', 'pr_auc': 0.03774573696739762, 'roc_auc': 0.44283665931592747, 'log_loss': 0.21373899189505333, 'withdrawn_mw_capture_at_10pct': 0.0, 'best_iteration': 1, 'reason': None}


## Val metrics, curves, save

In [5]:
save_bundle(metrics, model_obj=metrics.get('_model'), model_filename='model.joblib')

model → C:\Users\Magjun\Documents\Team-8-XternChallenge-\electrum\results\lightgbm\models\model.joblib


leaderboard → C:\Users\Magjun\Documents\Team-8-XternChallenge-\electrum\results\leaderboard.csv
status= ok wrote C:\Users\Magjun\Documents\Team-8-XternChallenge-\electrum\results\lightgbm
